In [1]:
# MÓDULO 1: Importando as bibliotecas necessárias
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import requests

print(" Bibliotecas importadas com sucesso! Prontos para começar a análise.")

 Bibliotecas importadas com sucesso! Prontos para começar a análise.


In [2]:
# MÓDULO 2: Extraindo dados do Ibovespa (Renda Variável)
print("Baixando histórico do Ibovespa (^BVSP) dos últimos 5 anos...")

ibov = yf.download('^BVSP', period='5y')[['Close']]
ibov.columns = ['Ibovespa'] 
ibov.index = pd.to_datetime(ibov.index).tz_localize(None) 

# O comando display gera uma tabela formatada no Jupyter!
display(ibov.head())

Baixando histórico do Ibovespa (^BVSP) dos últimos 5 anos...


[*********************100%***********************]  1 of 1 completed


,Ibovespa
Date,
2021-03-17,116549.0
2021-03-18,114835.0
2021-03-19,116222.0
2021-03-22,114979.0
2021-03-23,113262.0


In [3]:
# MÓDULO 3: Extraindo dados do Banco Central (CDI e IPCA) via API Pública
def buscar_dados_bcb(codigo_sgs, data_inicio):
    url = f'https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo_sgs}/dados?formato=json&dataInicial={data_inicio}'
    response = requests.get(url)
    df = pd.DataFrame(response.json())
    df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y')
    df['valor'] = df['valor'].astype(float)
    df.set_index('data', inplace=True)
    return df

# Definindo a data de 5 anos atrás até hoje para bater com o Ibovespa
data_inicio_analise = (pd.Timestamp.today() - pd.DateOffset(years=5)).strftime('%d/%m/%Y')

print("Consultando API do Banco Central...")
cdi_diario = buscar_dados_bcb(12, data_inicio_analise)   # Código 12 = CDI
ipca_mensal = buscar_dados_bcb(433, data_inicio_analise) # Código 433 = IPCA

print(" Dados macroeconômicos baixados com sucesso!")

Consultando API do Banco Central...
 Dados macroeconômicos baixados com sucesso!


In [4]:
# MÓDULO 4: Tratamento de Dados (Calculando a "Base 100" e unindo as bases)
print("Cruzando e nivelando as bases de dados...")

# 1. Ibovespa Acumulado
ibov['Ibov_Acumulado'] = (ibov['Ibovespa'] / ibov['Ibovespa'].iloc[0]) * 100

# 2. CDI Acumulado (Juros compostos diários)
cdi_diario['Fator_Diario'] = 1 + (cdi_diario['valor'] / 100)
cdi_diario['CDI_Acumulado'] = cdi_diario['Fator_Diario'].cumprod() * 100

# 3. IPCA Acumulado (Juros compostos mensais)
ipca_mensal['Fator_Mensal'] = 1 + (ipca_mensal['valor'] / 100)
ipca_mensal['IPCA_Acumulado'] = ipca_mensal['Fator_Mensal'].cumprod() * 100

# 4. Juntando as tabelas em um único DataFrame
df_final = pd.DataFrame(index=ibov.index)
df_final = df_final.join(ibov['Ibov_Acumulado'])
df_final = df_final.join(cdi_diario['CDI_Acumulado'])
df_final = df_final.join(ipca_mensal['IPCA_Acumulado']).ffill() # Preenche os dias sem inflação divulgada

df_final.dropna(inplace=True)

# Mostra como ficou a tabela final consolidada
display(df_final.tail())

Cruzando e nivelando as bases de dados...


,Ibov_Acumulado,CDI_Acumulado,IPCA_Acumulado
Date,,,
2026-03-11,157.846914,171.935006,131.674005
2026-03-12,153.828004,172.029796,131.674005
2026-03-13,152.427734,172.124638,131.674005
2026-03-16,154.334229,172.219532,131.674005
2026-03-17,155.719456,172.219532,131.674005


In [7]:
# MÓDULO 5: Data Visualization - O Gráfico Interativo com Plotly
import plotly.graph_objects as go

print("Gerando o Dashboard Interativo...")

# 1. Criando a figura (A nossa "prancheta" interativa)
fig = go.Figure()

# 2. Linha 1: Ibovespa (Renda Variável)
fig.add_trace(go.Scatter(
    x=df_final.index,
    y=df_final['Ibov_Acumulado'],
    name='Ibovespa (Renda Variável)',
    mode='lines',
    line=dict(color='#1f77b4', width=2) # Azul corporativo
))

# 3. Linha 2: CDI (Renda Fixa)
fig.add_trace(go.Scatter(
    x=df_final.index,
    y=df_final['CDI_Acumulado'],
    name='CDI (Renda Fixa)',
    mode='lines',
    line=dict(color='#2ca02c', width=2.5) # Verde
))

# 4. Linha 3: IPCA (Inflação)
fig.add_trace(go.Scatter(
    x=df_final.index,
    y=df_final['IPCA_Acumulado'],
    name='IPCA (Inflação)',
    mode='lines',
    line=dict(color='#d62728', width=2, dash='dash') # Vermelho tracejado (dash)
))

# 5. Estética do Gráfico (O layout executivo)
fig.update_layout(
    title='<b>Análise de Mercado: Ibovespa vs. CDI vs. Inflação (Últimos 5 Anos)</b><br><sup>Evolução do Poder de Compra (Base R$ 100)</sup>',
    xaxis_title='Tempo (Anos)',
    yaxis_title='Crescimento do Patrimônio',
    plot_bgcolor='white',
    hovermode='x unified', # A mágica: mostra os 3 valores juntos ao passar o mouse!
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    
    # Substituindo o plt.figtext pelas anotações (annotations) do Plotly
    annotations=[
        dict(
            x=0,
            y=-0.15, # Posição negativa joga o texto para baixo do eixo X
            xref='paper',
            yref='paper',
            text='Fonte: yfinance / API Banco Central do Brasil | Desenvolvido em Python',
            showarrow=False,
            font=dict(size=10, color='gray')
        )
    ]
)

# 6. Adicionando as linhas de grade sutis e o prefixo de R$ no eixo Y
fig.update_xaxes(showgrid=True, gridcolor='#ecf0f1')
fig.update_yaxes(showgrid=True, gridcolor='#ecf0f1', tickprefix="R$ ")

# 7. Exibindo o gráfico na tela
fig.show()

print("\n--- ANÁLISE CONCLUÍDA COM SUCESSO! ---")

Gerando o Dashboard Interativo...



--- ANÁLISE CONCLUÍDA COM SUCESSO! ---


In [8]:
# MÓDULO 4: Transformação de Dados e Cálculo de KPIs
print("Calculando indicadores de performance e normalizando ativos...")

# 1. Definindo os ativos que queremos comparar (Pode adicionar mais tickers aqui!)
tickers_comparacao = ['^BVSP', 'PETR4.SA', 'ITUB4.SA']
nomes_amigaveis = {'^BVSP': 'Ibovespa', 'PETR4.SA': 'Petrobras', 'ITUB4.SA': 'Itaú'}

# Baixando os dados de todos os ativos da lista de uma vez
dados_ativos = yf.download(tickers_comparacao, period='5y')['Close']
dados_ativos.index = pd.to_datetime(dados_ativos.index).tz_localize(None)

# 2. Criando o DataFrame Final e aplicando a Base 100 para todos
df_dashboard = pd.DataFrame(index=dados_ativos.index)

for ticker in tickers_comparacao:
    # Fórmula da Base 100: (Preço Atual / Primeiro Preço) * 100
    primeiro_preco = dados_ativos[ticker].dropna().iloc[0]
    df_dashboard[nomes_amigaveis[ticker]] = (dados_ativos[ticker] / primeiro_preco) * 100

# 3. Integrando CDI e IPCA acumulados (que calculamos no Bloco 3)
df_dashboard = df_dashboard.join(cdi_diario['CDI_Acumulado']).join(ipca_mensal['IPCA_Acumulado']).ffill()
df_dashboard.dropna(inplace=True)

# 4. Cálculo dos KPIs Finais (Quanto R$ 100 viraram hoje)
kpi_resultados = df_dashboard.iloc[-1].sort_values(ascending=False)

print("Processamento concluído. KPIs gerados!")
display(kpi_resultados.to_frame(name='Valor Final (Base R$ 100)'))

Calculando indicadores de performance e normalizando ativos...


[*********************100%***********************]  3 of 3 completed

Processamento concluído. KPIs gerados!


,Valor Final (Base R$ 100)
Petrobras,679.522506
Itaú,234.630085
CDI_Acumulado,172.219532
Ibovespa,155.074167
IPCA_Acumulado,131.674005


In [9]:
# MÓDULO 5: O Dashboard Interativo
import plotly.graph_objects as go

fig = go.Figure()

# Adicionando os ativos dinamicamente ao gráfico
cores = ['#1f77b4', '#ff7f0e', '#9467bd'] # Cores para as ações
for i, coluna in enumerate(nomes_amigaveis.values()):
    fig.add_trace(go.Scatter(x=df_dashboard.index, y=df_dashboard[coluna], name=coluna, line=dict(width=2, color=cores[i])))

# Adicionando CDI e Inflação com destaque
fig.add_trace(go.Scatter(x=df_dashboard.index, y=df_dashboard['CDI_Acumulado'], name='CDI (Renda Fixa)', line=dict(color='#2ca02c', width=3)))
fig.add_trace(go.Scatter(x=df_dashboard.index, y=df_dashboard['IPCA_Acumulado'], name='IPCA (Inflação)', line=dict(color='#d62728', width=2, dash='dash')))

# Configurações do Dashboard (KPIs em anotações)
melhor_ativo = kpi_resultados.index[0]
valor_max = kpi_resultados.iloc[0]

fig.update_layout(
    title='<b>MacroPerform: Dashboard de Performance Comparativa</b><br><sup>Evolução de R$ 100 investidos nos últimos 5 anos</sup>',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(l=50, r=50, t=100, b=100),
)

# Adicionando um "Card" de KPI dentro do gráfico
fig.add_annotation(
    xref="paper", yref="paper", x=0.02, y=0.95,
    text=f"🏆 <b>Líder de Rendimento:</b> {melhor_ativo} (R$ {valor_max:.2f})",
    showarrow=False, font=dict(size=14, color="green"),
    bgcolor="white", bordercolor="green", borderpad=4, opacity=0.9
)

fig.update_yaxes(tickprefix="R$ ", gridcolor='#f0f0f0')
fig.update_xaxes(gridcolor='#f0f0f0')

fig.show()